# 39장 · RAG 개선안의 품질과 비용 비교

개선안을 판단하려면 같은 평가 문항에 대한 기존 방식과 후보 방식의 결과를 나란히 놓습니다. 답이 좋아졌는지와 작업량이 늘었는지를 따로 기록합니다.

38장의 고정 평가 기록에서 기준선과 재정렬 후보의 답변, 검색 근거, 추가 작업량을 비교합니다. 어떤 실패가 개선됐는지에 따라 적용 여부를 결정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 기준선과 후보의 답변·출처 비교

아래 값은 제공 실험 기록의 dev-01에서 읽은 항목입니다. 자료와 질문, 후보 수 2, 최종 문서 수 1을 고정했습니다. 이 셀은 저장 기록을 대조하며 외부 모델의 응답시간을 측정하지 않습니다.


### 셀 01 · 기준선·후보의 답변 대조

baseline과 rerank의 문항 ID가 일치하는지 먼저 확인하세요.


In [1]:
baseline = {'question_id': 'dev-01',
 'answer': '도서 대출 기간은 14일입니다 [library-1:0]',
 'source_ids': ['library-1:0'],
 'rerank_calls': 0}
candidate = {'question_id': 'dev-01',
 'answer': '도서 대출 기간은 14일입니다 [library-1:0]',
 'source_ids': ['library-1:0'],
 'rerank_calls': 1}
assert baseline['question_id'] == candidate['question_id']
print(baseline['answer'] == candidate['answer'], baseline['source_ids'] == candidate['source_ids'])

True True


첫 True는 두 답변 문자열이 같다는 뜻이고, 둘째 True는 출처 ID 목록이 같다는 뜻입니다. 두 설정이 같은 오답을 반환할 수도 있으므로 답의 ‘14일’과 인용은 기대 근거에 따로 대조합니다. 개선안을 채택하려면 틀리던 질문이 새로 맞았는지, 이미 맞던 다른 질문에 손해가 생겼는지도 봐야 합니다. 두 설정 모두 맞힌 질문에서는 추가 호출이나 늘어난 처리 시간이 선택 근거가 될 수 있습니다.


### 같은 문항의 결과를 한 쌍으로 비교하기

<p>두 행의 question_id가 모두 dev-01임을 assert로 확인합니다. 그래야 결과가 달라졌을 때 질문을 바꾼 영향과 검색 방법을 바꾼 영향을 혼동하지 않습니다.</p>


### 과제 02 · 추가 작업량 계산

동일 문항에서 재순위 함수 호출이 몇 번 추가됐는지 계산하세요.


In [2]:
# 두 행의 rerank_calls 차이를 구하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(candidate[&#x27;rerank_calls&#x27;] - baseline[&#x27;rerank_calls&#x27;])</code></pre>
<p>실행 결과</p><pre><code>1</code></pre>
<p>한 번 더 호출했습니다. 이 값은 로컬 함수 실행 횟수여서 유료 모델 호출 횟수나 금액으로 바꿔 읽을 수 없습니다.</p>
</details>


## 작업량과 적용 기준 기록

대출·휴관·주차는 개발 세트입니다. 다음 표는 원문을 읽고 판정한 기록입니다. 공개 확인 문항인 반납함 질문에서는 “24시간” 뒤의 문장이 잘려 답변의 완결성이 별도 과제로 남았습니다.

<table><caption>개발 문항별 답과 재정렬 호출 횟수</caption><thead><tr><th scope="col">질문</th><th scope="col">기준선과 후보의 답</th><th scope="col">호출 횟수 (기준선 / 후보)</th></tr></thead><tbody><tr><td>대출 기간은 며칠인가요?</td><td>도서 대출 기간은 14일입니다 [library-1:0]</td><td>0 / 1</td></tr><tr><td>휴관 요일은 언제인가요?</td><td>도서관은 월요일에 휴관합니다. [library-2:0]</td><td>0 / 1</td></tr><tr><td>주차 요금은 얼마인가요?</td><td>제공된 자료에서 확인할 수 없습니다.</td><td>0 / 0</td></tr></tbody></table>


### 셀 03 · 실험 후보와 적용 설정 구분

실험에서 비교한 후보와 실제 적용할 설정을 각각 출력하세요.

셀 01에서 직접 읽은 것은 dev-01 한 건입니다. 채택 여부는 대출·휴관·주차 세 문항의 저장 결과를 함께 보고 판단합니다. 후보명을 rerank로 적는 것과 실제 서비스 방식을 baseline으로 유지하는 것은 별개입니다. 반납함 24시간 질문은 선택 후의 확인 사례입니다.


In [3]:
decision = {'selected_candidate': 'rerank', 'adopted_method': 'baseline', 'final_k': 1}
print(decision['selected_candidate'], decision['adopted_method'])

rerank baseline


검토 후보는 rerank이고 현재 유지한 방식은 baseline입니다. 서비스에 사용할 설정을 기록하는 필드는 adopted_method입니다. 문장 끝이 잘린 경우에는 작은 청크로 검색한 뒤 그 청크가 속한 더 긴 원문을 읽는 Parent–Child를 시험할 수 있습니다. 정확한 용어로 찾은 결과와 벡터 검색 결과를 결합하는 Hybrid는 검색 후보 자체가 부족할 때 검토할 방법입니다.


### 세 개발 문항의 답변과 추가 호출

<p>대출과 휴관 질문은 두 방법 모두 원문의 14일과 월요일을 답했습니다. 주차 요금은 두 방법 모두 답을 유보했습니다. 재정렬은 대출과 휴관에서 한 번씩 실행됐고 빈 후보를 받은 주차 질문에서는 실행되지 않았습니다.</p>

<p>시간 기록에서도 측정 범위를 확인하세요. dev-01에서 파일 검증을 포함한 한 번의 로컬 실행은 기준선 1.082ms, 재정렬 1.011ms였습니다. 반복 측정이나 예열이 없어 이 차이로 어느 쪽이 빠른지 결정할 수 없습니다.</p>


### 과제 04 · 개발 문항 세 건에서 늘어난 작업 계산

아래 기록은 동일한 세 개발 문항의 재정렬 함수 호출 수입니다. 두 방법의 합계와 차이를 계산하세요. 앞에서 확인한 품질 판정과 함께 이번에 적용할 방법을 한 문장으로 설명하세요.


In [ ]:
calls = {'baseline': [0, 0, 0], 'rerank': [1, 1, 0]}
# dev-01, dev-02, dev-03 순서입니다.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>calls = {&#x27;baseline&#x27;: [0, 0, 0], &#x27;rerank&#x27;: [1, 1, 0]}
base_total = sum(calls[&#x27;baseline&#x27;])
rerank_total = sum(calls[&#x27;rerank&#x27;])
print(base_total, rerank_total, rerank_total - base_total)</code></pre>
<p>실행 결과</p><pre><code>0 2 2</code></pre>
<p>개발 세 문항에서 재정렬로 바로잡은 답은 없고 함수 호출은 두 번 늘었습니다. 이 기록에서는 기준선을 유지합니다. 원래 후보 순위가 틀린 새 질문을 추가하면 판단이 달라질 수 있습니다.</p>
</details>


## 돌아보기

‘대출 질문 한 건에서 같다’와 ‘개발 세 건을 확인해 이번엔 보류한다’는 근거 범위가 다릅니다. 팀의 실험 기록에도 판단에 사용한 문항을 적어 두세요.
